# processing large ensembles

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
# import matplotlib.pyplot as plt
# from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
# import matplotlib as mpl
# import matplotlib.ticker as mticker
import netCDF4
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.util import add_cyclic_point
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
# from dask.distributed import Client
# client = Client(n_workers=4, threads_per_worker=2)
# client

In [4]:
from scipy import stats

In [5]:
# mpl.rcParams['font.family'] = 'Droid Sans'
# mpl.rcParams['font.size'] = 12
# # Edit axes parameters
# mpl.rcParams['axes.linewidth'] = 1.5
# # Tick properties
# mpl.rcParams['xtick.major.size'] = 5
# mpl.rcParams['xtick.minor.size'] = 3
# mpl.rcParams['xtick.major.width'] = 1
# mpl.rcParams['xtick.direction'] = 'out'
# mpl.rcParams['ytick.major.size'] = 5
# mpl.rcParams['ytick.minor.size'] = 3
# mpl.rcParams['ytick.major.width'] = 1
# mpl.rcParams['ytick.direction'] = 'out'

In [6]:
from dask.diagnostics import ProgressBar

### Functions needed for the analysis

In [10]:
from functions import preproc_funcs as funcs

In [11]:
from functions import xr_lowess

In [9]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [10]:
import glob
import multiprocessing as mp

In [12]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        # model_identifier = path_parts[7] + '_' + path_parts[9][:-1]
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


# Function to find the first file in each model's r1* directory
def find_all_files_extended(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files



## find the files for a single model

In [ ]:
model = 'E3SM-1-0'

In [13]:
import re

In [14]:
## cmip6
ts_pattern_hist = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical*.nc'
ts_pattern_ssp3 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_ssp370*.nc'
# ts_pattern_ssp3 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}*_historical_ssp370*.nc'

## cmip5
# ts_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical_rcp85_*.nc'

In [15]:
# ## cmip6
pr_pattern_hist = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical*.nc'
pr_pattern_ssp3 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_ssp370*.nc'
# pr_pattern_ssp3 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}*_historical_ssp370*.nc'

## cmip5
# pr_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical_rcp85_*.nc'

In [26]:
ts_hist_files = sorted(glob.glob(ts_pattern_hist))
ts_ssp3_files = sorted(glob.glob(ts_pattern_ssp3))

# ts_rcp85_files = sorted(glob.glob(ts_pattern_rcp85))

In [27]:
pr_hist_files = sorted(glob.glob(pr_pattern_hist))
pr_ssp3_files = sorted(glob.glob(pr_pattern_ssp3))

# pr_rcp85_files = sorted(glob.glob(pr_pattern_rcp85))

In [29]:
del ts_hist_files[3:6]
del pr_hist_files[3:6]

In [28]:
# assert len(pr_hist_files) == len(pr_ssp5_files)

In [30]:
len(ts_ssp3_files), ts_ssp3_files

(16,
 ['/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r10i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r11i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r12i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r16i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r17i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r18i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r19i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_ssp370_r1i1p1f2_g025_201501-210012.nc',
  '/g/data/su28/MMLEAv2/atmo

In [32]:
len(ts_hist_files), ts_hist_files

(16,
 ['/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r10i1p1f2_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r11i1p1f2_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r12i1p1f2_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r16i1p1f2_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r17i1p1f2_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r18i1p1f2_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r19i1p1f2_g025_185001-201412.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/tas/UKESM1-0-LL/tas_Amon_UKESM1-0-LL_historical_r1i1p1f2_g025_185001-201412.n

In [42]:
# len(pr_ssp5_files), pr_ssp5_files

In [33]:
def open_member_cmip6(hist_file, ssp_file, ens_idx):
    ds_hist = xr.open_mfdataset(hist_file, use_cftime=True)
    ds_ssp = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time
    ds = xr.concat([ds_hist, ds_ssp], dim='time')

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip6_alternate(ssp_file, ens_idx):
    ds = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip5(rcp_file, ens_idx):
    ds = xr.open_mfdataset(rcp_file, use_cftime=True)

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds

In [35]:
members = []
for i, (hf, sf) in enumerate(zip(ts_hist_files, ts_ssp3_files)):
    members.append(open_member_cmip6(hf, sf, i))


# members = []
# for i, rf in enumerate(ts_ssp3_files):
#     members.append(open_member_cmip6_alternate(rf, i))

ERROR 1: PROJ: proj_create_from_database: Open of /g/data/xp65/public/./apps/med_conda/envs/analysis3-25.09/share/proj failed


In [36]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 4GB
Dimensions:  (model: 16, time: 3012, lat: 72, lon: 144)
Coordinates:
  * time     (time) object 24kB 1850-01-16 00:00:00 ... 2100-12-16 00:00:00
  * lon      (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat      (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
    height   float64 8B 1.5
  * model    (model) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
Data variables:
    tas      (model, time, lat, lon) float64 4GB dask.array<chunksize=(1, 1, 72, 144), meta=np.ndarray>
    file_qf  (model, time) float64 386kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Attributes: (12/52)
    CDI:                       Climate Data Interface version 1.9.6 (http://m...
    history:                   Fri Jan 10 19:11:12 2020: cdo -O -b F64 -remap...
    source:                    UKESM1.0-LL (2018): \naerosol: UKCA-GLOMAP-mod...
    institution:               Met Office Hadley Centre, Fitzroy Road, Exeter...
    Conventions:               CF-1.7 CMIP-6.2
    activity_id:               CMIP
    ...                        ...
    cmor_version:              3.4.0
    tracking_id:               hdl:21.14100/9c4beee9-e389-48e7-8450-9def754a2a1f
    cmip6-ng:                  \ncontact = cmip6-archive@env.ethz.ch\ndescrip...
    original_file_names:       /net/atmos/data/cmip6/historical/Amon/tas/UKES...
    original_file_hash_codes:  078b2ee36c54f46f71c2fb3c5d9219b3b86766e5a05fe4...
    CDO:                       Climate Data Operators version 1.9.6 (http://m...

In [37]:
tas = ds["tas"]
# tas = tas.sel(time = slice('1850-01-01', '2100-12-31'))
tas = tas.chunk(dict(model=8, time=120, lat=50, lon=50))

In [38]:
tas_ann = tas.resample(time = 'AS-JUN').mean('time')

In [ ]:
with ProgressBar():
    tas_ann_loaded = tas_ann.load()

[                                        ] | 0% Completed | 536.66 ms

In [ ]:
members = []
for i, (hf, sf) in enumerate(zip(pr_hist_files, pr_ssp3_files)):
    members.append(open_member_cmip6(hf, sf, i))


# members = []
# for i, rf in enumerate(pr_ssp3_files):
#     members.append(open_member_cmip6_alternate(rf, i))

In [ ]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 5GB
Dimensions:    (model: 21, time: 3012, bnds: 2, lat: 72, lon: 144)
Coordinates:
  * time       (time) object 24kB 1850-01-01 00:00:00 ... 2100-12-01 00:00:00
  * lon        (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat        (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * model      (model) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (model, time, bnds) object 1MB dask.array<chunksize=(1, 1, 2), meta=np.ndarray>
    pr         (model, time, lat, lon) float64 5GB dask.array<chunksize=(1, 1, 72, 144), meta=np.ndarray>
Attributes: (12/28)
    CDI:                       Climate Data Interface version 2.3.0 (https://...
    Conventions:               CF-1.7
    source:                    E3SM Atmosphere Model
    institution:               LLNL (Lawrence Livermore National Laboratory, ...
    ne:                        30
    fv_nphys:                  2
    ...                        ...
    remap_version:             5.0.1
    NCO:                       netCDF Operators version 5.0.1 (Homepage = htt...
    nco_openmp_thread_number:  2
    map_file:                  /global/u2/s/strandwg/proj/proc/2E3SM/map_ne30...
    input_file:                /pscratch/sd/s/strandwg/E3SMv2/v2.LR.historica...
    CDO:                       Climate Data Operators version 2.3.0 (https://...

In [ ]:
pr = ds["pr"]
# pr = pr.sel(time = slice('1850-01-01', '2100-12-31'))
pr = pr.chunk(dict(model=8, time=120, lat=50, lon=50))

In [ ]:
pr_ann = pr.resample(time = 'AS-JUN').mean('time')

In [ ]:
with ProgressBar():
    pr_ann_loaded = pr_ann.load()

[#######                                 ] | 19% Completed | 45.16 ss

[########################################] | 100% Completed | 222.79 s


In [ ]:
pr_ann_loaded

<xarray.DataArray 'pr' (model: 21, time: 252, lat: 72, lon: 144)> Size: 439MB
array([[[[1.30552552e-06, 1.30561388e-06, 1.30579129e-06, ...,
          1.53571361e-06, 1.53454730e-06, 1.53403623e-06],
         [1.51117873e-06, 1.50975248e-06, 1.50688935e-06, ...,
          1.55629643e-06, 1.56047040e-06, 1.56229957e-06],
         [2.01962848e-06, 2.01716505e-06, 2.01222014e-06, ...,
          2.45472835e-06, 2.46237296e-06, 2.46572321e-06],
         ...,
         [4.77256420e-06, 4.77045234e-06, 4.76621315e-06, ...,
          4.60800773e-06, 4.61499260e-06, 4.61805381e-06],
         [4.42078775e-06, 4.41974466e-06, 4.41765071e-06, ...,
          4.40085705e-06, 4.40236522e-06, 4.40302609e-06],
         [3.69426897e-06, 3.69297890e-06, 3.69038912e-06, ...,
          3.33811593e-06, 3.34179342e-06, 3.34340498e-06]],

        [[1.38515525e-06, 1.38537117e-06, 1.38580464e-06, ...,
          1.61418278e-06, 1.61354939e-06, 1.61327183e-06],
         [1.33126653e-06, 1.33095173e-06, 1.33031980e-06, ...,
          1.52818224e-06, 1.52839618e-06, 1.52848994e-06],
         [1.45245446e-06, 1.45233350e-06, 1.45209065e-06, ...,
          1.60655290e-06, 1.60702786e-06, 1.60723600e-06],
...
         [1.44720942e-05, 1.44542347e-05, 1.44183851e-05, ...,
          1.18613553e-05, 1.18859904e-05, 1.18967871e-05],
         [1.23500973e-05, 1.23519283e-05, 1.23556042e-05, ...,
          1.07544796e-05, 1.07529392e-05, 1.07522641e-05],
         [1.42839095e-05, 1.42861355e-05, 1.42906042e-05, ...,
          1.23853789e-05, 1.23795356e-05, 1.23769750e-05]],

        [[2.60005236e-06, 2.60040077e-06, 2.60110022e-06, ...,
          3.04677111e-06, 3.04551034e-06, 3.04495787e-06],
         [2.20984430e-06, 2.21191121e-06, 2.21606042e-06, ...,
          2.66591815e-06, 2.66318735e-06, 2.66199065e-06],
         [1.68988981e-06, 1.69135869e-06, 1.69430724e-06, ...,
          2.21680175e-06, 2.21283142e-06, 2.21109141e-06],
         ...,
         [1.15656403e-05, 1.15627314e-05, 1.15568924e-05, ...,
          1.02513183e-05, 1.02509337e-05, 1.02507653e-05],
         [1.22809385e-05, 1.22866050e-05, 1.22979804e-05, ...,
          1.20890118e-05, 1.20707276e-05, 1.20627150e-05],
         [1.39109637e-05, 1.39123396e-05, 1.39151018e-05, ...,
          1.39268793e-05, 1.39244565e-05, 1.39233948e-05]]]])
Coordinates:
  * lon      (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat      (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * model    (model) int64 168B 0 1 2 3 4 5 6 7 8 ... 12 13 14 15 16 17 18 19 20
  * time     (time) object 2kB 1849-06-01 00:00:00 ... 2100-06-01 00:00:00
Attributes:
    long_name:     precipitation_flux
    units:         kg m-2 s-1
    cell_methods:  time: mean

In [ ]:
tas_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/smiles/{model}_ts.nc')
pr_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/smiles/{model}_pr.nc')

In [56]:
# tas_anom = funcs.calc_anom(tas_ann_loaded, tas_ann_loaded.sel(time = slice('1960', '1990')))
# tas_det = funcs.detrend_rolling_window(tas_anom, window_size=15)

In [57]:
# pr_det = funcs.calc_anom(pr_ann_loaded, pr_ann_loaded.sel(time = slice('1960', '1990')))
# pr_det = funcs.detrend_rolling_window(pr_det, window_size=15)
# pr_std = pr_det/pr_det.std('time')